# Cosmology inference

When planes carry **redshifts** and the model carries a **cosmology**, the
cosmological parameters become free parameters you can sample. This fits a two-source
system and infers `Om0`, `w0`, and an unknown source redshift.

:::{admonition} float64 is required
:class: important
Cosmological inference runs the float64 pipeline: `JAX_ENABLE_X64=1` **and**
`likelihood_precision="float64"` (this notebook sets `jax_enable_x64` at import).
:::

In [ ]:
%matplotlib inline
import jax
jax.config.update("jax_enable_x64", True)
import numpy as np
from jax import numpy as jnp
import optax
import tensorflow_probability.substrates.jax as tfp
import matplotlib as mpl
from matplotlib import pyplot as plt
from corner import corner
tfd = tfp.distributions

import gigalens
from gigalens.simulator import SimulatorConfig
from gigalens.jax.cosmo import w0waCDM_Cosmo as Cosmo
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.mass.shear import Shear
from gigalens.jax.profiles.light.sersic import SersicEllipse
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.scene_simulator import SceneSimulator
from gigalens.jax.scene_prob_model import ImageData, ProbModel
from gigalens.jax.inference import ModellingSequence
print('jax', jax.__version__, '| devices', jax.devices())

## Model: lens, two source planes, free cosmology

A cosmology is a `Component` wrapping a cosmology profile, attached via `cosmo=`;
`z_source_ref` is **required**. Here `Om0` and `w0` are sampled while `H0`, `wa`, `k`
are fixed constants. With a cosmology present, **every** plane uses `redshift=` (not
`deflection_ratio=`), and a redshift can itself be a prior — the second source's
redshift is inferred.

In [ ]:
z_lens, z_s1 = 0.5, 1.2
cosmo = Component(Cosmo(z_lens=z_lens, z_source_ref=z_s1),
                  dict(H0=70.0, Om0=tfd.Uniform(0.01, 0.99), w0=tfd.Uniform(-2.0, -1/3), wa=0.0, k=0.0))
epl = Component(EPL(), dict(
    theta_E=tfd.LogNormal(jnp.log(1.25), 0.25), gamma=tfd.TruncatedNormal(2, 0.25, 1, 3),
    e1=tfd.Normal(0, 0.1), e2=tfd.Normal(0, 0.1), center_x=0.1, center_y=0.0))
shear = Component(Shear(), dict(gamma1=tfd.Normal(0, 0.05), gamma2=tfd.Normal(0, 0.05)))
lens_light = Component(SersicEllipse(), dict(
    R_sersic=tfd.LogNormal(jnp.log(1.0), 0.15), n_sersic=tfd.Uniform(2, 6),
    e1=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3), e2=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
    center_x=0.1, center_y=0.0, Ie=tfd.LogNormal(jnp.log(500.0), 0.3)))
def sersic_source():
    return dict(R_sersic=tfd.LogNormal(jnp.log(0.25), 0.15), n_sersic=tfd.Uniform(0.5, 4),
                e1=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5), e2=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5),
                center_x=tfd.Normal(0, 0.25), center_y=tfd.Normal(0, 0.25),
                Ie=tfd.LogNormal(jnp.log(150.0), 0.5))
source_1 = Component(SersicEllipse(), sersic_source())
source_2 = Component(SersicEllipse(), sersic_source())

model = LensModel([
    Plane(redshift=z_lens, mass=[epl, shear], light=[lens_light]),
    Plane(redshift=z_s1, light=[source_1]),
    Plane(redshift=tfd.Uniform(1.2, 3.5), light=[source_2]),   # unknown source redshift, inferred
], cosmo=cosmo)
names = list(model.z_param_names)
print('free parameters:', model.num_free_params)
print('z-columns:', names)

## Data: simulate from a known truth

The observation is generated by rendering the model at injected truth values (target
`Om0`, `w0`, and source redshift) and adding noise, so the posterior can be checked
against ground truth.

In [ ]:
background_rms, exp_time = 0.1, 200
root = gigalens.__path__[0]
kernel = np.load(f'{root}/assets/psf.npy').astype(np.float32)
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=60, supersample=1, kernel=kernel,
                             likelihood_precision='float64')

# truth: prior draw with the target values injected for the inferred quantities
truth_unique = model.prior.sample(seed=jax.random.PRNGKey(0))
truth_unique = dict(truth_unique)
truth_unique['planes/2/geometry/redshift'] = jnp.asarray(2.3)
truth_unique['cosmo/Om0'] = jnp.asarray(0.3)
truth_unique['cosmo/w0'] = jnp.asarray(-1.0)
truth = np.array([float(truth_unique[n]) for n in names])

sim = SceneSimulator(model, sim_config)
clean = np.asarray(sim.simulate(model.to_params(truth_unique)))
err_map = np.sqrt(background_rms**2 + np.clip(clean, 0, np.inf) / exp_time)
np.random.seed(1)
observed_img = clean + np.random.normal(scale=err_map)

ds = ImageData(observed_img, sim_config, background_rms=background_rms, exp_time=exp_time, sees='all')
prob = ProbModel(model, ds, mode='forward')
seq = ModellingSequence(prob)
plt.imshow(observed_img, norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=10), origin='lower')
plt.colorbar(); plt.title('Observed'); plt.show()

## Inference: MAP → SVI → HMC

:::{admonition} `wCDM_Cosmo` vs `w0waCDM_Cosmo`
:class: note
This demo uses `w0waCDM_Cosmo` (adds `wa`) and samples `Om0`/`w0`. The point-source
tutorial uses `wCDM_Cosmo` and samples `H0`. Both require `z_source_ref`.
:::

In [ ]:
def to_constrained(z_rows):
    xb = model.bijector.forward(jnp.asarray(z_rows).reshape(-1, len(names)))
    return np.stack([np.asarray(xb[n]).reshape(-1) for n in names], axis=1)

opt = optax.adabelief(1e-2, b1=0.95, b2=0.99)
best_z, best_lp, map_chisqs = seq.MAP(opt, seed=42, num_steps=1000, output_type='best')
best_z = np.asarray(jax.device_get(best_z))
print('MAP log-post: %.4g' % float(best_lp))

In [ ]:
opt = optax.adabelief(1e-4, b1=0.95, b2=0.99)
qz, loss_hist = seq.SVI(best_z, opt, n_vi=400, num_steps=500)
plt.plot(np.asarray(loss_hist).reshape(-1)); plt.xlabel('step'); plt.ylabel('-ELBO'); plt.title('SVI loss'); plt.show()

In [ ]:
# Rebuild the SVI surrogate from host arrays: under JAX 0.10 the sharded qz trips a
# mesh (Manual vs Explicit) clash inside HMC's pmapped sampler. device_get strips the
# sharding tag while keeping the SVI-learned mean + covariance.
qz = tfd.MultivariateNormalFullCovariance(
    loc=np.asarray(jax.device_get(qz.mean())),
    covariance_matrix=np.asarray(jax.device_get(qz.covariance())))
samples = seq.HMC(qz, num_burnin_steps=250, num_results=1000)
rhat = np.asarray(tfp.mcmc.potential_scale_reduction(samples, independent_chain_ndims=2))
ess = np.asarray(tfp.mcmc.effective_sample_size(samples, cross_chain_dims=[1, 2]))
print('max R-hat: %.3f | min ESS: %.0f' % (np.nanmax(rhat), np.nanmin(ess)))
n_params = samples.shape[-1]
post = to_constrained(np.asarray(samples).reshape(-1, n_params))

## Posterior: cosmology + inferred source redshift

The sampled quantities live in `z_param_names` under `cosmo/Om0`, `cosmo/w0`, and
`planes/2/geometry/redshift`; recover them with `model.bijector.forward(...)`.

In [ ]:
cosmo_paths = ['cosmo/Om0', 'cosmo/w0', 'planes/2/geometry/redshift']
cosmo_labels = [r'$\Omega_{m,0}$', r'$w_0$', r'$z_{\rm source,2}$']
idx = [names.index(p) for p in cosmo_paths]
fig = corner(post[:, idx], labels=cosmo_labels, truths=truth[idx], show_titles=True, title_fmt='.3f')
fig.suptitle('Cosmology + source redshift'); plt.show()